<style>
.reveal { font-family: 'Segoe UI', system-ui, sans-serif; font-size: 1.05em; }
.reveal h2 { color: #0D2240; border-bottom: 2.5px solid #1A7A9A; padding-bottom: .3em; }
.reveal h3 { color: #1A7A9A; }
.reveal .slides section { text-align: left; }
.reveal pre { font-size: .75em; box-shadow: none; border-left: 3px solid #1A7A9A; }
.defn { background:#EAF6FA; border-left:4px solid #1A7A9A; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.nota { background:#FFF8E1; border-left:4px solid #C8961E; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.alerta { background:#FDE8E8; border-left:4px solid #C0392B; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
</style>

## Proceso de Poisson
### T2.1 · Modelado de Sistemas bajo Incertidumbre
Universidad de los Andes · Ingeniería Industrial

## Agenda
1. Motivación: Línea 123 — tres preguntas, tres distribuciones
2. Definición axiomática del Proceso de Poisson
3. Distribución de N(t) — los cuatro tipos de probabilidad
4. Tiempos entre llegadas — Exponencial y propiedad sin memoria
5. Tiempo hasta el n-ésimo evento — Erlang
6. **Dualidad Poisson–Erlang:** P(N(t)≥n) = P(Sₙ≤t)
7. **Superposición (merging)** — hospital de urgencias
8. **Descomposición (thinning/splitting)** — triaje y correo
9. Competencia entre exponenciales

## Motivación: Línea de Emergencias 123

La línea 123 de una ciudad recibe en promedio **λ = 4 llamadas por minuto**. Tres preguntas naturales requieren tres herramientas distintas:

| Pregunta | Herramienta | Distribución |
|---|---|---|
| ¿Cuántas llamadas en los próximos 2 min? | Conteo $N(t)$ | $\text{Poisson}(\lambda t)$ |
| ¿Cuánto tiempo entre llamadas consecutivas? | Tiempo entre eventos $T_i$ | $\text{Exp}(\lambda)$ |
| ¿Cuánto hasta acumular 5 llamadas? | Tiempo al $n$-ésimo evento $S_n$ | $\text{Erlang}(n, \lambda)$ |

**Características que hacen aplicable el modelo Poisson:**
- Eventos ocurren **uno a la vez** (no simulténeamente)
- La tasa promedio $\lambda$ es **estable** en el período de observación
- Lo ocurrido en el pasado **no altera** la probabilidad futura

**Más ejemplos en IE:** fallas de máquinas (por semana), defectos en una línea de producción, pedidos a un CD (por día), pacientes que llegan a urgencias (por hora).

## Definición axiomática
El proceso {N(t), t ≥ 0} es un **Proceso de Poisson** con tasa λ > 0 si:

1. **N(0) = 0** — inicia sin eventos
2. **Incrementos independientes** — eventos en intervalos disjuntos son independientes
3. **Incrementos estacionarios** — la distribución de N(t+s) − N(t) solo depende de s
4. **Regularidad:** P(un evento en (t, t+h]) = λh + o(h)
5. **No simultaneidad:** P(dos o más eventos en (t, t+h]) = o(h)

<div class='nota'>
Las condiciones 4 y 5 implican que en un intervalo muy pequeño ocurre a lo sumo un evento con probabilidad proporcional a λh.
</div>

## Distribución de N(t) — Los Cuatro Tipos de Probabilidad

<div class='defn'>

$N(t) \sim \text{Poisson}(\lambda t)$ con $P(N(t)=k) = \dfrac{(\lambda t)^k e^{-\lambda t}}{k!}$, $\;k=0,1,2,\ldots$

**Media = Varianza = λt** (diagnóstico empírico de un proceso de Poisson)

</div>

| Pregunta | Fórmula | Python (`mu = λt`) |
|---|---|---|
| **Exactamente** $k$ | $P(N=k)$ | `poisson.pmf(k, mu)` |
| **A lo sumo** $k$ | $P(N \leq k) = \sum_{j=0}^{k} P(N=j)$ | `poisson.cdf(k, mu)` |
| **Más de** $k$ | $P(N > k) = 1 - P(N \leq k)$ | `1 - poisson.cdf(k, mu)` |
| **Al menos** $k$ | $P(N \geq k) = 1 - P(N \leq k-1)$ | `1 - poisson.cdf(k-1, mu)` |
| **Entre** $a$ y $b$ | $P(a \leq N \leq b) = P(N \leq b) - P(N \leq a-1)$ | `poisson.cdf(b, mu) - poisson.cdf(a-1, mu)` |

<div class='nota'>

**Truco:** "al menos $k$" = complemento de "a lo sumo $k-1$". No confundir $P(N>k)$ con $P(N \geq k)$.

</div>

In [ ]:
from scipy.stats import poisson
import numpy as np
import matplotlib.pyplot as plt

lam = 4.0   # λ = 4 llamadas/min  (línea 123)
ts  = [0.5, 1, 2]
ks  = np.arange(0, 22)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, t in zip(axes, ts):
    mu  = lam * t
    pmf = poisson.pmf(ks, mu)
    ax.bar(ks, pmf, color='#1A7A9A', alpha=0.85, edgecolor='white')
    # resaltar cola derecha (más de la media)
    cola = ks > mu
    ax.bar(ks[cola], pmf[cola], color='#C8961E', alpha=0.85, edgecolor='white',
           label=f'k > {mu:.0f}')
    ax.axvline(mu, color='#C62828', lw=2, ls='--', label=f'Media λt = {mu:.0f}')
    ax.set_title(f'N({t} min) ~ Poisson({mu:.0f})', fontsize=10)
    ax.set_xlabel('k  (llamadas)'); ax.set_ylabel('P(N(t) = k)')
    ax.legend(fontsize=8)
plt.suptitle('Línea 123 — N(t) para distintos intervalos de tiempo',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
from scipy.stats import poisson

# Línea 123: λ=4/min, intervalo t=2 min → N(2) ~ Poisson(8)
lam, t = 4.0, 2.0
mu = lam * t

print(f"Línea 123:  λ = {lam}/min,  t = {t} min  →  N({t}) ~ Poisson(λt = {mu:.0f})")
print("─" * 62)
print(f"  {'Pregunta':<40} {'Resultado':>10}")
print("─" * 62)

k = 10
print(f"  P(N(2) = {k})   — exactamente {k} llamadas   {poisson.pmf(k, mu):>10.4f}")

k = 8
print(f"  P(N(2) ≤  {k})  — a lo sumo {k} llamadas    {poisson.cdf(k, mu):>10.4f}")

k = 10
print(f"  P(N(2) > {k})   — más de {k} llamadas        {1 - poisson.cdf(k, mu):>10.4f}")

k = 5
print(f"  P(N(2) ≥  {k})  — al menos {k} llamadas      {1 - poisson.cdf(k-1, mu):>10.4f}")

a, b = 5, 12
print(f"  P({a} ≤ N(2) ≤ {b}) — entre {a} y {b} llamadas    {poisson.cdf(b,mu) - poisson.cdf(a-1,mu):>10.4f}")

print(f"  P(N(2) = 0)   — ninguna llamada           {poisson.pmf(0, mu):>10.6f}")
print("─" * 62)
print(f"  E[N(2)] = {mu:.0f}  |  Var[N(2)] = {mu:.0f}  |  SD = {mu**0.5:.3f}")

## Tiempos entre Llegadas — Exponencial y Sin Memoria

<div class='defn'>

Si $\{N(t)\}$ es PP(λ), los tiempos entre eventos consecutivos $T_1, T_2, \ldots$ son **i.i.d. Exp(λ)**:

$$P(T > t) = P(N(t) = 0) = e^{-\lambda t}, \quad E[T] = 1/\lambda, \quad CV = 1$$

</div>

**Propiedad sin memoria** (exclusiva de la Exponencial entre distribuciones continuas):

$$P(T > s + t \mid T > s) = P(T > t) = e^{-\lambda t}$$

El sistema **no recuerda** cuánto lleva esperando — base de los modelos $M/M/c$ (markovianos).

<div class='nota'>

**Ejemplo — cajero bancario** μ = 6 clientes/h (media 10 min), $S \sim \text{Exp}(6)$:
$$P(S > 15\text{ min}) = e^{-6 \times 0.25} = e^{-1.5} \approx 0.223$$
$$P(S > 15\text{ min} \mid S > 10\text{ min}) = P(S > 5\text{ min}) = e^{-6 \times 0.0833} = e^{-0.5} \approx 0.607$$
Después de 10 min atendiendo, la probabilidad de 5 min más es la misma que desde cero.

</div>

## Tiempo hasta el n-ésimo Evento — Erlang y Dualidad

<div class='defn'>

$S_n = T_1 + \cdots + T_n$ (suma de $n$ exponenciales i.i.d.) sigue una **Erlang(n, λ)**:

$$f_{S_n}(t) = \frac{\lambda^n t^{n-1} e^{-\lambda t}}{(n-1)!}, \quad E[S_n] = \frac{n}{\lambda}, \quad CV(S_n) = \frac{1}{\sqrt{n}} < 1$$

</div>

**Dualidad Poisson–Erlang** — dos formas de calcular lo mismo:

$$\boxed{P(N(t) \geq n) = P(S_n \leq t)}$$

*"Llegan al menos $n$ eventos antes de $t$"* es **equivalente** a *"el $n$-ésimo evento ocurre antes de $t$"*

| Pregunta en la línea 123 | Vía Poisson | Vía Erlang |
|---|---|---|
| ¿5ª llamada antes de 2 min? | $P(N(2) \geq 5)$ con $\mu=8$ | $P(S_5 \leq 2)$ con Erlang$(5,4)$ |
| ¿3 autos atendidos en 12 min? | $P(N(0.2) \geq 3)$ con $\mu=2$ | $P(S_3 \leq 0.2)$ con Erlang$(3,10)$ |

La dualidad permite elegir **la vía de cálculo más conveniente** según el enunciado.

In [ ]:
from scipy.stats import poisson, gamma
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

print("─" * 66)
print("  Dualidad:  P(N(t) ≥ n)  =  P(Sₙ ≤ t)")
print("─" * 66)

casos = [
    ("Línea 123: ¿5ª llamada antes de 2 min?",       4,   5, 2.0),
    ("Gasolina:  ¿3 autos atendidos en 12 min?",     10,  3, 0.2),
    ("C.datos:   ¿4 fallas en la 1ª semana?",        0.5, 4, 7.0),
    ("Farmacia:  ¿despacho >8 min? Erlang(3,0.75)", 0.75, 3, 8.0),
]
for desc, lam, n, t in casos:
    mu       = lam * t
    p_pois   = 1 - poisson.cdf(n - 1, mu)
    p_erlang = gamma.cdf(t, a=n, scale=1/lam)
    print(f"\n  {desc}")
    print(f"  λ={lam}, n={n}, t={t}  →  λt = {mu:.2f}")
    print(f"  P(N({t}) ≥ {n})  via Poisson({mu:.2f})     = {p_pois:.5f}")
    print(f"  P(S_{n} ≤ {t})   via Erlang({n},{lam})   = {p_erlang:.5f}  ✓")
print("─" * 66)

# Visualización: densidades Erlang + PMF coloreado para dualidad
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

lam_v, t_v, n_v = 4, 2, 5
t_range = np.linspace(0, 4, 400)
for nv, color in zip([1, 3, 5, 10], ['#C62828', '#C8961E', '#1A7A9A', '#0D2240']):
    axes[0].plot(t_range, gamma.pdf(t_range, a=nv, scale=1/lam_v),
                 color=color, lw=2, label=f'Erlang({nv}, 4)')
axes[0].axvline(t_v, color='gray', ls='--', lw=1.5, label=f't = {t_v} min')
axes[0].set_xlabel('Tiempo (min)'); axes[0].set_ylabel('Densidad')
axes[0].set_title('Erlang(n, λ=4): densidad para distintos n')
axes[0].legend(fontsize=9)

mu_v   = lam_v * t_v
k_range = np.arange(0, 22)
col_pmf = ['#C62828' if k >= n_v else '#1A7A9A' for k in k_range]
axes[1].bar(k_range, poisson.pmf(k_range, mu_v), color=col_pmf, edgecolor='white', alpha=0.85)
p_val = 1 - poisson.cdf(n_v - 1, mu_v)
axes[1].set_xlabel('k'); axes[1].set_ylabel('P(N(2) = k)')
axes[1].set_title(f'Dualidad: P(N(2)≥{n_v}) = P(S₅≤2) = {p_val:.4f}')
axes[1].legend([Patch(color='#C62828'), Patch(color='#1A7A9A')],
               [f'N(2) ≥ {n_v}  ← zona de dualidad', f'N(2) < {n_v}'], fontsize=9)
plt.tight_layout(); plt.show()

## Superposición (Merging)

<div class='defn'>

Si $N_1(t), \ldots, N_k(t)$ son PP independientes con tasas $\lambda_1, \ldots, \lambda_k$, su **unión** es:

$$N(t) = N_1(t) + \cdots + N_k(t) \;\sim\; \text{PP}\!\left(\sum_{i=1}^k \lambda_i\right)$$

</div>

**Caso: Sistema de Urgencias Hospitalarias**

Un hospital recibe pacientes desde tres fuentes independientes:

| Fuente | Tasa |
|---|---|
| Ambulancias | $\lambda_1 = 2$/h |
| Pacientes caminando | $\lambda_2 = 8$/h |
| Remisiones de clínicas | $\lambda_3 = 1.5$/h |
| **Flujo total (PP)** | **$\lambda = 11.5$/h** |

**Preguntas:** ¿P(más de 15 en 1h)? ¿P(entre 10 y 14)? ¿Tiempo promedio entre llegadas?

In [ ]:
from scipy.stats import poisson
import numpy as np
import matplotlib.pyplot as plt

lam1, lam2, lam3 = 2.0, 8.0, 1.5
lam   = lam1 + lam2 + lam3
t     = 1.0
mu    = lam * t

print(f"Superposición — Urgencias: λ = {lam1}+{lam2}+{lam3} = {lam}/h")
print(f"N(1h) ~ Poisson({mu})")
print("─" * 54)
print(f"  P(N(1) = 12)          = {poisson.pmf(12, mu):.4f}")
print(f"  P(N(1) ≤ 10)          = {poisson.cdf(10, mu):.4f}")
print(f"  P(N(1) > 15)          = {1-poisson.cdf(15, mu):.4f}  ← riesgo de sobrecarga")
print(f"  P(10 ≤ N(1) ≤ 14)     = {poisson.cdf(14,mu)-poisson.cdf(9,mu):.4f}")
print(f"  E[N(1)] = {mu:.1f}  |  SD = {mu**0.5:.3f}")
print(f"  Tiempo medio entre llegadas = {60/lam:.2f} min")

fig, ax = plt.subplots(figsize=(9, 4))
k_range = np.arange(0, 26)
col = ['#C62828' if k > 15 else '#1A7A9A' for k in k_range]
ax.bar(k_range, poisson.pmf(k_range, mu), color=col, edgecolor='white', alpha=0.85)
ax.axvline(mu, color='#C8961E', lw=2.5, ls='--', label=f'Media = {mu}')
ax.set_xlabel('Pacientes por hora'); ax.set_ylabel('Probabilidad')
ax.set_title(f'Superposición: N(1h) ~ Poisson({mu})  [3 fuentes independientes]')
from matplotlib.patches import Patch
ax.legend([plt.Line2D([0],[0], color='#C8961E', lw=2, ls='--'),
           Patch(color='#C62828'), Patch(color='#1A7A9A')],
          ['Media = 11.5', f'Sobrecarga P(N>15) = {1-poisson.cdf(15,mu):.3f}', 'Normal'],
          fontsize=9)
plt.tight_layout(); plt.show()

## Descomposición (Thinning / Splitting)

<div class='defn'>

Si $N(t)$ es PP(λ) y cada evento se clasifica en categoría $i$ con probabilidad $p_i$ (**independientemente**), los procesos resultantes son **PP independientes** con tasas $\lambda p_i$:

$$N_i(t) \;\sim\; \text{PP}(\lambda\, p_i), \qquad \text{todos independientes entre sí}$$

</div>

**Caso 1 — Triaje en urgencias** ($\lambda = 11.5$/h):

| Nivel | $p_i$ | $\lambda p_i$ |
|---|---|---|
| Alta urgencia | 0.10 | 1.15/h |
| Media urgencia | 0.35 | 4.025/h |
| Baja urgencia | 0.55 | 6.325/h |

**Caso 2 — Servidor de correo** ($\lambda = 20$/min, $p_{\text{spam}} = 0.15$):
- $N_{\text{spam}}(t) \sim \text{PP}(3)$ y $N_{\text{legít}}(t) \sim \text{PP}(17)$ — **independientes**
- $P(\text{2 spam y 15 legítimos en 1 min}) = P(N_s=2) \times P(N_l=15)$  ← ¡por independencia!

In [ ]:
from scipy.stats import poisson
import numpy as np
import matplotlib.pyplot as plt

# --- Caso 1: Triaje de urgencias ---
lam_h = 11.5
niveles = [('Alta',  0.10), ('Media', 0.35), ('Baja',  0.55)]
t2 = 2.0

print("Descomposición — Triaje urgencias (λ=11.5/h)")
print("─" * 56)
for nivel, p in niveles:
    lam_n = lam_h * p
    print(f"  {nivel:6}: λ × p = {lam_h} × {p} = {lam_n:>5.3f}/h  →  PP({lam_n:.3f})")
lam_alta = lam_h * 0.10
print(f"\n  En t=2h, N_alta(2) ~ Poisson({lam_alta*t2:.2f})")
print(f"  P(N_alta(2h) = 0)  = {poisson.pmf(0, lam_alta*t2):.4f}  (ningún paciente crítico)")
print(f"  P(N_alta(2h) ≥ 3)  = {1 - poisson.cdf(2, lam_alta*t2):.4f}")
print(f"  E[N_alta(2h)]      = {lam_alta*t2:.2f} pacientes")

# --- Caso 2: Servidor de correo ---
lam_m, p_spam = 20.0, 0.15
lam_s = lam_m * p_spam
lam_l = lam_m * (1 - p_spam)
print(f"\nDescomposición — Servidor de correo (λ={lam_m}/min)")
print("─" * 56)
print(f"  Spam (p={p_spam}):       λ_spam  = {lam_s}/min  →  PP({lam_s})")
print(f"  Legítimo (p={1-p_spam}):  λ_legit = {lam_l}/min  →  PP({lam_l})")
print(f"\n  P(N_spam(1min) = 0) = {poisson.pmf(0, lam_s):.4f}  (sin spam)")
print(f"  P(N_spam(1min) ≤ 5) = {poisson.cdf(5, lam_s):.4f}")
p_joint = poisson.pmf(2, lam_s) * poisson.pmf(15, lam_l)
print(f"  P(2 spam Y 15 legít) = {poisson.pmf(2,lam_s):.4f} × {poisson.pmf(15,lam_l):.4f} = {p_joint:.5f}  (independencia)")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# PMFs de los 3 sub-flujos de triaje en t=2h
colors = ['#C62828', '#C8961E', '#1A7A9A']
k_range = np.arange(0, 20)
for (nivel, p), color in zip(niveles, colors):
    mu_n = lam_h * p * t2
    axes[0].plot(k_range, poisson.pmf(k_range, mu_n), 'o-', color=color,
                 lw=1.5, ms=4, label=f'{nivel}: Pois({mu_n:.2f})')
axes[0].set_xlabel('k (pacientes en 2h)'); axes[0].set_ylabel('Probabilidad')
axes[0].set_title('PMFs de los 3 sub-flujos — triaje (t=2h)')
axes[0].legend(fontsize=9)

# PMF de spam vs legítimo en 1 min
k_s = np.arange(0, 12)
k_l = np.arange(0, 30)
axes[1].bar(k_s, poisson.pmf(k_s, lam_s), color='#C62828', alpha=0.8,
            edgecolor='white', label=f'Spam  Pois({lam_s})')
ax2 = axes[1].twinx()
ax2.bar(k_l + 0.4, poisson.pmf(k_l, lam_l), color='#1A7A9A', alpha=0.5,
        edgecolor='white', width=0.8, label=f'Legítimo  Pois({lam_l})')
axes[1].set_xlabel('k'); axes[1].set_ylabel('P — Spam', color='#C62828')
ax2.set_ylabel('P — Legítimo', color='#1A7A9A')
axes[1].set_title('Spam vs Legítimo (1 min) — independientes')
axes[1].legend(loc='upper right', fontsize=9)
ax2.legend(loc='upper center', fontsize=9)
plt.tight_layout(); plt.show()

## Competencia entre Exponenciales

<div class='defn'>

Si $T_1 \sim \text{Exp}(\lambda_1), \ldots, T_n \sim \text{Exp}(\lambda_n)$ son independientes:

$$\min(T_1, \ldots, T_n) \sim \text{Exp}\!\left(\sum_{i=1}^n \lambda_i\right), \qquad P(T_i \text{ es el mínimo}) = \frac{\lambda_i}{\sum_j \lambda_j}$$

</div>

**Aplicación 1 — Máquinas en paralelo:**
Tasas de falla $\lambda_1=0.01$, $\lambda_2=0.02$, $\lambda_3=0.015$ fallas/h
→ Primera falla del sistema ~ Exp(0.045), media = **22.2 h**
→ La máquina 2 falla primero con prob. $0.02/0.045 = \mathbf{44.4\%}$

**Aplicación 2 — Motor de simulación de eventos discretos (SED):**
En una cola $M/M/2$ con $\lambda=5$, $\mu_1=\mu_2=3$ (ambos servidores ocupados), hay **tres relojes compitiendo**:

| Reloj | Tasa | P(gana) |
|---|---|---|
| Nueva llegada | 5 | 5/11 = 45.5% |
| Servidor 1 termina | 3 | 3/11 = 27.3% |
| Servidor 2 termina | 3 | 3/11 = 27.3% |

El próximo evento ocurre en tiempo Exp(11), media = 5.45 min.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import expon
np.random.seed(42)

# --- Máquinas en paralelo ---
lam1, lam2, lam3 = 0.01, 0.02, 0.015
lam_tot = lam1 + lam2 + lam3
n_sim   = 60_000
T1 = np.random.exponential(1/lam1, n_sim)
T2 = np.random.exponential(1/lam2, n_sim)
T3 = np.random.exponential(1/lam3, n_sim)
T_min         = np.minimum(np.minimum(T1, T2), T3)
primera_falla = np.argmin(np.column_stack([T1, T2, T3]), axis=1)

print("Competencia — Máquinas en paralelo (tasas de falla por hora)")
print("─" * 58)
print(f"  λ₁={lam1}, λ₂={lam2}, λ₃={lam3}  →  λ_total={lam_tot}")
print(f"  min(T₁,T₂,T₃) ~ Exp({lam_tot})  →  media teórica = {1/lam_tot:.1f} h")
print(f"  media simulada = {T_min.mean():.2f} h")
print()
for i, (lam_i, nombre) in enumerate(zip([lam1, lam2, lam3], ['Máquina 1', 'Máquina 2', 'Máquina 3'])):
    p_teo = lam_i / lam_tot
    p_sim = np.mean(primera_falla == i)
    print(f"  P({nombre} falla primero): teórico={p_teo:.3f}  simulado={p_sim:.3f}")

# --- Cola M/M/2 ---
lam_mm, mu1, mu2 = 5, 3, 3
tot_mm = lam_mm + mu1 + mu2
print(f"\n  Cola M/M/2: λ={lam_mm}, μ₁=μ₂={mu1}  (ambos servidores ocupados)")
print(f"  Próximo evento ~ Exp({tot_mm})  →  media = {1/tot_mm:.4f} h")
print(f"  P(nueva llegada)        = {lam_mm}/{tot_mm} = {lam_mm/tot_mm:.4f}")
print(f"  P(algún servidor acaba) = {mu1+mu2}/{tot_mm} = {(mu1+mu2)/tot_mm:.4f}")

# Visualización
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(T_min, bins=70, density=True, color='#1A7A9A', edgecolor='white', alpha=0.85,
        label='Simulación')
t_r = np.linspace(0, 300, 500)
ax.plot(t_r, expon.pdf(t_r, scale=1/lam_tot), '#C62828', lw=2.5, ls='--',
        label=f'Exp({lam_tot}) teórica')
ax.set_xlabel('Tiempo a la primera falla (horas)'); ax.set_ylabel('Densidad')
ax.set_title(f'min(T₁,T₂,T₃) ~ Exp({lam_tot}) — verificación por simulación')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Verificación empírica: simular PP(λ=4) y comprobar distribuciones
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import poisson

np.random.seed(7)
lam_real  = 4       # λ = 4 llamadas/min (línea 123)
n_eventos = 500

interarrivals = np.random.exponential(1/lam_real, size=n_eventos)
llegadas      = np.cumsum(interarrivals)

# Contar en ventanas de 1 minuto
T_max    = llegadas[-1]
ventanas = np.arange(0, T_max, 1.0)
conteos  = [np.sum((llegadas >= v) & (llegadas < v+1)) for v in ventanas]
lam_est  = np.mean(conteos)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribución empírica vs Poisson(λ̂)
k_vals = np.arange(0, max(conteos)+1)
axes[0].hist(conteos, bins=range(max(conteos)+2), density=True,
             color='#1A7A9A', alpha=0.7, edgecolor='white', label='Empírico')
axes[0].plot(k_vals, poisson.pmf(k_vals, lam_est), 'o-',
             color='#C62828', lw=2, ms=5, label=f'Poisson({lam_est:.2f})')
axes[0].set_xlabel('N(1 minuto)'); axes[0].set_ylabel('Densidad')
axes[0].set_title('Conteos/minuto: empírico vs Poisson'); axes[0].legend()

# Q-Q plot: tiempos entre llegadas vs Exp teórica
probs = (np.arange(1, n_eventos+1) - 0.5) / n_eventos
teor  = stats.expon.ppf(probs, scale=1/lam_est)
axes[1].scatter(np.sort(interarrivals), teor, alpha=0.3, color='#1A7A9A', s=8)
lim = max(np.sort(interarrivals)[-5], teor[-5])
axes[1].plot([0, lim], [0, lim], 'r--', lw=2, label='y = x')
axes[1].set_xlabel('Cuantiles empíricos'); axes[1].set_ylabel('Cuantiles Exp teórica')
axes[1].set_title('Q-Q Plot: tiempos entre llegadas vs Exp(4)'); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"  λ̂ estimada = {lam_est:.3f}  (verdadera = {lam_real})")

## Conclusiones y Preguntas de Práctica

**El ecosistema Poisson:**

| Componente | Distribución | Pregunta |
|---|---|---|
| Conteo en $[0,t]$ | $\text{Poisson}(\lambda t)$ | ¿Cuántos eventos en el intervalo? |
| Entre eventos | $\text{Exp}(\lambda)$ | ¿Cuánto esperar al próximo? |
| Hasta $n$-ésimo | $\text{Erlang}(n,\lambda)$ | ¿Cuándo se acumula el $n$-ésimo? |
| Fusión de $k$ fuentes | $\text{PP}(\sum \lambda_i)$ | ¿Flujo combinado? |
| División por categoría | $\text{PP}(\lambda p_i)$ | ¿Flujo por tipo? |
| Mínimo de exponenciales | $\text{Exp}(\sum \lambda_i)$ | ¿Quién "gana" la carrera? |

**Preguntas de práctica:**

1. Línea 123 (λ=4/min): calcule P(N(3)=15), P(N(3)≥10), P(5≤N(3)≤11).
2. Por dualidad: ¿P(5ª llamada antes de 1.5 min) = P(N(1.5)≥5)? Calcule ambas y verifique.
3. Urgencias: se añade una 4ª fuente (traslados, λ₄=3/h). ¿Nuevo P(N(1h)>15)? ¿Cuánto cambió?
4. Servidor de correo (λ=20/min, p_spam=0.15): si la tasa total sube a 30/min con p_spam=0.10, ¿cómo cambia P(N_spam(1)=0)?
5. Propiedad sin memoria: una máquina lleva 8 h funcionando con T~Exp(0.1 fallas/h). ¿Cuánto tiempo adicional se espera hasta la falla? ¿Por qué?